# 04 Data Transformation & Feature Engineering
* Create time and delivery features on clean operational tables.
* Aggregate financial and payment records to the order level.
* Construct Star Schema analytical tables (`fact_orders`, `dim_customers`, `dim_products`, `dim_sellers`) for SQL and Power BI.

In [1]:
import os
import urllib.parse
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from sqlalchemy import create_engine

# Load credentials
load_dotenv()

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "3306")
DB_NAME = os.getenv("DB_NAME")

# Database Connection
ENCODED_PASSWORD = urllib.parse.quote_plus(DB_PASSWORD)
engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{ENCODED_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

print("✅ Connected to MySQL for Data Transformation!")

✅ Connected to MySQL for Data Transformation!


### 2. Loading Cleaned Tables from MySQL


In [2]:
df_orders = pd.read_sql("SELECT * FROM olist_orders_clean", con=engine)
df_customers = pd.read_sql("SELECT * FROM olist_customers_clean", con=engine)
df_products = pd.read_sql("SELECT * FROM olist_products_clean", con=engine)
df_items = pd.read_sql("SELECT * FROM olist_order_items_clean", con=engine)
df_payments = pd.read_sql("SELECT * FROM olist_order_payments_clean", con=engine)
df_reviews = pd.read_sql("SELECT * FROM olist_order_reviews_clean", con=engine)
df_sellers = pd.read_sql("SELECT * FROM olist_sellers_clean", con=engine)

print("All cleaned tables loaded successfully into memory!")

All cleaned tables loaded successfully into memory!


### 3. Time Feature Engineering
Extracting temporal attributes (Year, Month, Day, Hour, Day of Week) from `order_purchase_timestamp` to enable time-series slicing and trend analysis.

In [3]:
# Ensure timestamp is datetime type
df_orders["order_purchase_timestamp"] = pd.to_datetime(df_orders["order_purchase_timestamp"])

# Extract Date & Time Features
df_orders["purchase_year"] = df_orders["order_purchase_timestamp"].dt.year
df_orders["purchase_month"] = df_orders["order_purchase_timestamp"].dt.month
df_orders["purchase_month_name"] = df_orders["order_purchase_timestamp"].dt.strftime("%b")
df_orders["purchase_day"] = df_orders["order_purchase_timestamp"].dt.day
df_orders["purchase_hour"] = df_orders["order_purchase_timestamp"].dt.hour
df_orders["purchase_weekday"] = df_orders["order_purchase_timestamp"].dt.day_name()

print("Time features extracted successfully!")
print(df_orders[["order_purchase_timestamp", "purchase_year", "purchase_month_name", "purchase_weekday", "purchase_hour"]].head(3))

Time features extracted successfully!
  order_purchase_timestamp  purchase_year purchase_month_name  \
0      2017-09-13 08:59:02           2017                 Sep   
1      2017-04-26 10:53:06           2017                 Apr   
2      2018-01-14 14:33:31           2018                 Jan   

  purchase_weekday  purchase_hour  
0        Wednesday              8  
1        Wednesday             10  
2           Sunday             14  


### 4. Delivery KPI Feature Engineering
Calculating delivery lead times, estimated delivery gaps, and delayed order flags.

In [4]:
# Ensure delivery columns are datetime
for col in ["order_delivered_customer_date", "order_estimated_delivery_date"]:
    df_orders[col] = pd.to_datetime(df_orders[col])

# 1. Delivery Time (Days) = Delivered Date - Purchase Date
df_orders["delivery_time_days"] = (
    df_orders["order_delivered_customer_date"] - df_orders["order_purchase_timestamp"]
).dt.total_seconds() / (24 * 3600)

# Round to 2 decimal places
df_orders["delivery_time_days"] = df_orders["delivery_time_days"].round(2)

# 2. Estimated Delivery Gap (Days) = Estimated Date - Delivered Date
# Positive = Delivered earlier than estimated, Negative = Delivered late
df_orders["estimated_delivery_gap_days"] = (
    df_orders["order_estimated_delivery_date"] - df_orders["order_delivered_customer_date"]
).dt.total_seconds() / (24 * 3600)

df_orders["estimated_delivery_gap_days"] = df_orders["estimated_delivery_gap_days"].round(2)

# 3. Delay Flag (1 if actual delivery date > estimated delivery date, else 0)
df_orders["is_delayed"] = (
    df_orders["order_delivered_customer_date"] > df_orders["order_estimated_delivery_date"]
).astype(int)

print("Delivery features engineered successfully!")
print(df_orders[["order_id", "delivery_time_days", "estimated_delivery_gap_days", "is_delayed"]].head(3))

Delivery features engineered successfully!
                           order_id  delivery_time_days  \
0  00010242fe8c5a6d1ba2dd792cb16214                7.61   
1  00018f77f2f0320c557190d7a144bdd3               16.22   
2  000229ec398224ef6ca0657da4fc703e                7.95   

   estimated_delivery_gap_days  is_delayed  
0                         8.01           0  
1                         2.33           0  
2                        13.44           0  


### 5. Financial Aggregations (Order-Level Items & Freight)
Aggregates line-item prices, freight costs, and total item counts per order.

In [5]:
# Aggregate items to order level
df_order_financials = (
    df_items.groupby("order_id")
    .agg(
        total_price=("price", "sum"),
        total_freight=("freight_value", "sum"),
        item_count=("order_item_id", "count"),
    )
    .reset_index()
)

# Calculate total order value (price + freight)
df_order_financials["total_order_value"] = (
    df_order_financials["total_price"] + df_order_financials["total_freight"]
).round(2)

print("Financial metrics aggregated successfully.")
print(df_order_financials.head(3))

Financial metrics aggregated successfully.
                           order_id  total_price  total_freight  item_count  \
0  00010242fe8c5a6d1ba2dd792cb16214         58.9          13.29           1   
1  00018f77f2f0320c557190d7a144bdd3        239.9          19.93           1   
2  000229ec398224ef6ca0657da4fc703e        199.0          17.87           1   

   total_order_value  
0              72.19  
1             259.83  
2             216.87  


### 6. Payment Aggregations
Aggregate total payments, maximum installments, and primary payment method per order.

In [6]:
# Identify primary payment type per order (payment method with highest value)
df_payment_primary = (
    df_payments.sort_values(by=["order_id", "payment_value"], ascending=[True, False])
    .groupby("order_id")
    .first()
    .reset_index()[["order_id", "payment_type"]]
    .rename(columns={"payment_type": "primary_payment_type"})
)

# Aggregate payment values and installments
df_payment_totals = (
    df_payments.groupby("order_id")
    .agg(
        total_payment_value=("payment_value", "sum"),
        max_installments=("payment_installments", "max"),
        payment_sequential_count=("payment_sequential", "count"),
    )
    .reset_index()
)

# Merge payment summaries
df_order_payments_agg = pd.merge(
    df_payment_totals, df_payment_primary, on="order_id", how="left"
)

df_order_payments_agg["total_payment_value"] = df_order_payments_agg[
    "total_payment_value"
].round(2)

print("Payment metrics aggregated successfully.")
print(df_order_payments_agg.head(3))

Payment metrics aggregated successfully.
                           order_id  total_payment_value  max_installments  \
0  00010242fe8c5a6d1ba2dd792cb16214                72.19                 2   
1  00018f77f2f0320c557190d7a144bdd3               259.83                 3   
2  000229ec398224ef6ca0657da4fc703e               216.87                 5   

   payment_sequential_count primary_payment_type  
0                         1          credit_card  
1                         1          credit_card  
2                         1          credit_card  


### 7. Build Master Fact Table (`fact_orders`)
Merges orders with customer details, order financials, payment summaries, and review scores.

In [7]:
# 1. Merge orders with customer location
fact_orders = pd.merge(
    df_orders,
    df_customers[["customer_id", "customer_unique_id", "customer_city", "customer_state"]],
    on="customer_id",
    how="left",
)

# 2. Merge financials
fact_orders = pd.merge(fact_orders, df_order_financials, on="order_id", how="left")

# 3. Merge payment metrics
fact_orders = pd.merge(fact_orders, df_order_payments_agg, on="order_id", how="left")

# 4. Merge review score (take average score per order if multiple reviews exist)
df_review_scores = (
    df_reviews.groupby("order_id")
    .agg(review_score=("review_score", "mean"))
    .reset_index()
)
df_review_scores["review_score"] = df_review_scores["review_score"].round(2)

fact_orders = pd.merge(fact_orders, df_review_scores, on="order_id", how="left")

print("fact_orders table created successfully.")
print(f"Shape: {fact_orders.shape}")
print(fact_orders[["order_id", "customer_unique_id", "total_order_value", "primary_payment_type", "review_score"]].head(3))

fact_orders table created successfully.
Shape: (99441, 29)
                           order_id                customer_unique_id  \
0  00010242fe8c5a6d1ba2dd792cb16214  871766c5855e863f6eccc05f988b23cb   
1  00018f77f2f0320c557190d7a144bdd3  eb28e67c4c0b83846050ddfb8a35d051   
2  000229ec398224ef6ca0657da4fc703e  3818d81c6709e39d06b2738a8d3a2474   

   total_order_value primary_payment_type  review_score  
0              72.19          credit_card           5.0  
1             259.83          credit_card           4.0  
2             216.87          credit_card           5.0  


### 8. Build Dimension Tables
Creating dim_customers, dim_products, and dim_sellers with aggregated performance metrics.

In [8]:
# --- 8.1 dim_customers ---
dim_customers = (
    fact_orders.groupby("customer_unique_id")
    .agg(
        total_orders=("order_id", "nunique"),
        total_spend=("total_order_value", "sum"),
        avg_order_value=("total_order_value", "mean"),
        first_purchase_date=("order_purchase_timestamp", "min"),
        latest_purchase_date=("order_purchase_timestamp", "max"),
        customer_city=("customer_city", "first"),
        customer_state=("customer_state", "first"),
    )
    .reset_index()
)

dim_customers["total_spend"] = dim_customers["total_spend"].round(2)
dim_customers["avg_order_value"] = dim_customers["avg_order_value"].round(2)

# --- 8.2 dim_products ---
# Join items with product catalog to summarize sales by product
df_product_sales = (
    df_items.groupby("product_id")
    .agg(
        units_sold=("order_item_id", "count"),
        total_revenue=("price", "sum"),
        avg_price=("price", "mean"),
        avg_freight=("freight_value", "mean"),
    )
    .reset_index()
)

dim_products = pd.merge(
    df_products, df_product_sales, on="product_id", how="left"
).fillna({"units_sold": 0, "total_revenue": 0, "avg_price": 0, "avg_freight": 0})

dim_products["total_revenue"] = dim_products["total_revenue"].round(2)
dim_products["avg_price"] = dim_products["avg_price"].round(2)
dim_products["avg_freight"] = dim_products["avg_freight"].round(2)

# --- 8.3 dim_sellers ---
dim_sellers = (
    df_items.groupby("seller_id")
    .agg(
        total_orders_handled=("order_id", "nunique"),
        items_sold=("order_item_id", "count"),
        total_revenue=("price", "sum"),
        avg_item_price=("price", "mean"),
    )
    .reset_index()
)

dim_sellers = pd.merge(df_sellers, dim_sellers, on="seller_id", how="left").fillna(
    {"total_orders_handled": 0, "items_sold": 0, "total_revenue": 0, "avg_item_price": 0}
)

dim_sellers["total_revenue"] = dim_sellers["total_revenue"].round(2)
dim_sellers["avg_item_price"] = dim_sellers["avg_item_price"].round(2)

print("Dimension tables created successfully.")
print(f"dim_customers shape: {dim_customers.shape}")
print(f"dim_products shape:  {dim_products.shape}")
print(f"dim_sellers shape:   {dim_sellers.shape}")

Dimension tables created successfully.
dim_customers shape: (96096, 8)
dim_products shape:  (32951, 14)
dim_sellers shape:   (3095, 8)


### 9. Export Analytical Tables to MySQL
Saving fact_orders and dimension tables into MySQL as the Gold Data Layer for analytical queries and Power BI dashboarding.

In [9]:
transformed_tables = {
    "fact_orders": fact_orders,
    "dim_customers": dim_customers,
    "dim_products": dim_products,
    "dim_sellers": dim_sellers,
}

for table_name, df in transformed_tables.items():
    df.to_sql(name=table_name, con=engine, if_exists="replace", index=False)
    print(f"Exported: '{table_name}' ({len(df):,} rows)")

print("\nAll transformed Gold-layer tables successfully saved to MySQL.")

Exported: 'fact_orders' (99,441 rows)
Exported: 'dim_customers' (96,096 rows)
Exported: 'dim_products' (32,951 rows)
Exported: 'dim_sellers' (3,095 rows)

All transformed Gold-layer tables successfully saved to MySQL.


In [10]:
import os

# Set up transformed output directory
transformed_dir = os.path.join("..", "data", "transformed")
os.makedirs(transformed_dir, exist_ok=True)

transformed_tables = {
    "fact_orders": fact_orders,
    "dim_customers": dim_customers,
    "dim_products": dim_products,
    "dim_sellers": dim_sellers,
}

# Export transformed tables to CSV
for table_name, df in transformed_tables.items():
    csv_path = os.path.join(transformed_dir, f"{table_name}.csv")
    df.to_csv(csv_path, index=False)
    print(f" Saved to CSV: '{table_name}.csv' ({len(df):,} rows)")

print("\n All transformed Gold-layer tables saved in 'data/transformed/' folder!")

 Saved to CSV: 'fact_orders.csv' (99,441 rows)
 Saved to CSV: 'dim_customers.csv' (96,096 rows)
 Saved to CSV: 'dim_products.csv' (32,951 rows)
 Saved to CSV: 'dim_sellers.csv' (3,095 rows)

 All transformed Gold-layer tables saved in 'data/transformed/' folder!


## 10. Transformation Summary

* **Feature Engineering:** Extracted temporal attributes (year, month, weekday, hour) and calculated delivery KPIs (lead time in days, delivery gap, late flag).
* **Order Aggregations:** Summarized item-level financials (price, freight, total order value) and payment methods at the `order_id` grain.
* **Data Modeling:** Constructed a Star Schema consisting of 1 central Fact table (`fact_orders`) and 3 Dimension tables (`dim_customers`, `dim_products`, `dim_sellers`).
* **Database Export:** Persisted all 4 Gold-layer analytical tables into MySQL.